# 13. Robustez, generalización y pruebas adversas

**Fases del guía metodológica cubiertas: 19 (Robustez, generalización y pruebas adversas)**



## 19.1 Pruebas de entrada

Comprobamos el comportamiento del pipeline con entradas degeneradas: nulos, tipos
erróneos, categorías desconocidas y valores extremos. La API valida el esquema
(fase 21.3); aquí probamos el pipeline directamente.

### 19.1.1 Carga del pipeline

Cargamos el pipeline final entrenado en la fase 16. A partir de aquí, cada prueba de
entrada consistirá en tomar una fila real de test, modificarla de forma degenerada y
comprobar que la predicción no rompe (y, cuando corresponda, que el imputador y los
encoders manejan la situación con sensatez).


In [1]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import joblib, json, numpy as np, pandas as pd
from src.data.load_data import load_processed
from src.features.build_features import add_domain_features

pipeline = joblib.load(ROOT / "models" / "final_model.joblib")
d = load_processed()
Xte = add_domain_features(d["X_test"])
cols = Xte.columns.tolist()
print("Pipeline cargado. Features:", len(cols))


Pipeline cargado. Features: 37



### 19.1.2 Categoría desconocida, valores extremos y control

Ejecutamos tres pruebas: (1) una **categoría nunca vista** (`school="ZZ"`), que el
OneHotEncoder con `handle_unknown="ignore"` debe tolerar; (2) un **valor extremo**
(`absences=999`), que los árboles manejan sin problema al no asumir escala; y (3) una
fila normal como **control** para comparar. Si la probabilidad cambia de forma brusca
frente al control, lo anotamos como comportamiento esperado o sospechoso.


In [2]:

# 1) Categoría desconocida (OneHot handle_unknown='ignore')
x_unk = Xte.iloc[:1].copy()
x_unk["school"] = "ZZ"
p_unk = pipeline.predict_proba(x_unk)[0, 1]
print("Categoría desconocida -> proba:", round(p_unk, 4), "(no rompe)")

# 2) Valores extremos (absences=999)
x_ext = Xte.iloc[:1].copy()
x_ext["absences"] = 999
print("Valor extremo absences -> proba:", round(pipeline.predict_proba(x_ext)[0, 1], 4))

# 3) Fila normal (control)
print("Control                -> proba:", round(pipeline.predict_proba(Xte.iloc[:1])[0, 1], 4))


Categoría desconocida -> proba: 0.8824 (no rompe)
Valor extremo absences -> proba: 0.8341


Control                -> proba: 0.8754



### 19.1.3 Nulos

El pipeline incluye imputadores (`median` para numéricas, `most_frequent` para
categóricas) ajustados solo con train. Inyectamos `NaN` en una numérica y en una
categórica y comprobamos que la inferencia no falla. Esto cubre el caso de que en
producción llegue un registro incompleto (aunque la API lo impida, el pipeline debe ser
defensivo).


In [3]:

# 4) Nulos: el imputador del pipeline los maneja (mediana/most_frequent)
x_nan = Xte.iloc[:1].copy()
x_nan.loc[x_nan.index[0], "absences"] = np.nan
x_nan.loc[x_nan.index[0], "school"] = np.nan
print("Nulos -> proba:", round(pipeline.predict_proba(x_nan)[0, 1], 4))


Nulos -> proba: 0.8812



## 19.2 Pruebas de cambio (dataset shift)

Simulamos **covariate shift** ligero: añadimos ruido gaussiano a `age` y `absences`
y comparamos la distribución de probabilidades predichas. Con ello estimamos la
sensibilidad del modelo a cambios en la población.

### 19.2.1 Shift simulado y correlación de predicciones

Creamos una copia de test con ruido (edad +/-0.5, ausencias +/-2) y comparamos las
probabilidades predichas antes/después. Si la correlación es alta (> 0.95) y el cambio
medio pequeño, el modelo es estable frente a perturbaciones moderadas; si no, sería
sensible a drift y debería monitorizarse con más atención.


In [4]:

rng = np.random.default_rng(42)
X_shift = Xte.copy()
X_shift["age"] = X_shift["age"] + rng.normal(0, 0.5, len(X_shift))
X_shift["absences"] = X_shift["absences"] + rng.normal(0, 2, len(X_shift)).clip(0, None)

p_base = pipeline.predict_proba(Xte)[:, 1]
p_shift = pipeline.predict_proba(X_shift)[:, 1]
print("Correlación probas base vs shift:", round(np.corrcoef(p_base, p_shift)[0, 1], 4))
print("Cambio medio en proba:", round(np.mean(np.abs(p_base - p_shift)), 4))


Correlación probas base vs shift:

 0.9963
Cambio medio en proba: 0.0099



### 19.2.2 Estabilidad entre semillas

El split está fijo, pero el **modelo** depende de la semilla de entrenamiento (los
árboles usan aleatoriedad). Entrenamos 5 pipelines con semillas distintas sobre el mismo
train y evaluamos en validation. Una desviación pequeña (< 0.02) indica que el modelo no
depende críticamente de la semilla -> reproducible.


In [5]:

# Estabilidad entre semillas: 5 pipelines con semillas distintas sobre el MISMO split
from src.models.train_model import make_pipeline, _lightgbm
from sklearn.metrics import roc_auc_score
from src.data.load_data import load_processed
from src.features.build_features import add_domain_features
d = load_processed()
Xva = add_domain_features(d["X_val"]); yva = d["y_val"]

aucs = []
for seed in [1, 7, 42, 123, 2024]:
    p = make_pipeline(_lightgbm(seed))
    p.fit(add_domain_features(d["X_train"]), d["y_train"])
    aucs.append(roc_auc_score(yva, p.predict_proba(Xva)[:, 1]))
print("ROC-AUC val por semilla:", [round(a, 4) for a in aucs])
print("Media:", round(np.mean(aucs), 4), "+/-", round(np.std(aucs), 4))


ROC-AUC val por semilla: [0.8324, 0.8383, 0.8248, 0.8234, 0.8283]
Media: 0.8294 +/- 0.0054



## 19.3 Pruebas técnicas

- **Reproducibilidad de entrenamiento**: misma semilla -> mismos artefactos (verificado en `tests/`).
- **Compatibilidad entrenamiento/servicio**: la API usa el mismo `final_model.joblib`
  (notebook 15) y los tests de `tests/test_api_validation.py` cubren la validación de inputs.
- **Latencia**: batch de 100 predicciones en la API ~ milisegundos (modelo de 100 árboles).
- **Memoria**: pipeline < 5 MB en disco.

### 19.3.1 Latencia y tamaño del artefacto

Medimos el tiempo de 50 inferencias completas (pipeline + preprocesado) y el tamaño en
disco del artefacto. Para un uso batch anual estos números son irrelevantes, pero
documentan el coste operativo si algún día se quisiera servir en tiempo real.


In [6]:

import time, joblib
t0 = time.time()
for _ in range(50):
    pipeline.predict_proba(Xte)
print(f"50 predicciones sobre {len(Xte)} filas: {time.time()-t0:.2f}s "
      f"({(time.time()-t0)/50:.2f} ms/predicción)")
print("Tamaño del artefacto (MB):", round(joblib.dump(pipeline, 'tmp.joblib')[0] and
      (ROOT/'tmp.joblib').stat().st_size / 1e6, 2))
(ROOT/'tmp.joblib').unlink(missing_ok=True)


50 predicciones sobre 133 filas: 3.07s (0.06 ms/predicción)


Tamaño del artefacto (MB): 4.66



## 19.4 Diagnóstico visual de la generalización (curva de aprendizaje, ROC/PR y calibración)

La fase 19 (robustez y generalización) es el lugar natural para la **evidencia gráfica**
de que el modelo generaliza: la curva de aprendizaje (20 puntos), las curvas ROC y PR
en test, la calibración, la matriz de confusión y la distribución de probabilidades.
Estas figuras **no cambian ninguna decisión** (el test ya se usó en la fase 17); solo
verifican visualmente que el entrenamiento fue correcto.

### 19.4.1 Preparación: carga del pipeline, metadatos y calibrador

Cargamos el pipeline final, sus metadatos (umbral congelado) y el calibrador sigmoidal
guardado en la fase 16. Generamos las probabilidades crudas y calibradas sobre test,
exactamente como las produce la API en producción.


In [7]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from src.evaluation.metrics import (
    compute_metrics, calibration_summary, roc_curve_data, pr_curve_data,
    learning_curve_data,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Datos: train (para la curva de aprendizaje) y test (para ROC/PR/calibracion)
d = load_processed()
Xtr = add_domain_features(d["X_train"]); ytr = d["y_train"]
Xte = add_domain_features(d["X_test"]);  yte = d["y_test"]
print("Train:", Xtr.shape, "| Test:", Xte.shape)

# Pipeline final y metadatos
pipeline = joblib.load(ROOT / "models" / "final_model.joblib")
meta = json.loads((ROOT / "models" / "final_model_metadata.json").read_text(encoding="utf-8"))
umbral = float(meta["threshold"])
print("Modelo:", meta["model_name"], "| umbral de coste:", round(umbral, 3))

# Calibrador sigmoidal (para comparar crudo vs calibrado)
cal_path = ROOT / "models" / "final_calibrator.joblib"
calibrator = joblib.load(cal_path) if cal_path.exists() else None
print("Calibrador cargado:", calibrator is not None)

# Probabilidades sobre test
p_raw = pipeline.predict_proba(Xte)[:, 1]
p_cal = calibrator.predict_proba(p_raw.reshape(-1, 1))[:, 1] if calibrator is not None else p_raw
print("Probabilidades test (raw y calibradas) listas.")


Train: (396, 37) | Test: (133, 37)
Modelo: RandomForest | umbral de coste: 0.34
Calibrador cargado: True


Probabilidades test (raw y calibradas) listas.



### 19.4.2 Curva de aprendizaje con 20 puntos

La **curva de aprendizaje** responde a la pregunta: *¿cuántos datos necesita este modelo
y cómo se comporta con los que tiene?* Se entrena el pipeline sobre subconjuntos de
tamaño creciente del train (20 puntos, del 5 % al 100 %) y, para cada tamaño, se evalúa
con **StratifiedKFold(5)** tanto en el propio subconjunto como en la parte de validación
de cada fold.

Interpretación esperada de un **modelo sano**: la curva de train empieza alta y
desciende suavemente; la de validation empieza baja y **asciende hasta una meseta**;
ambas convergen con una brecha (gap) pequeña y estable. Un gap que **crece** con los
datos delataría overfitting; unas curvas planas y bajas, underfitting.

Usamos `learning_curve_data` de `src/evaluation/metrics.py` con `n_points=20` y
persistimos los valores en `reports/learning_curve.json` para análisis posterior.


In [8]:

# Curva de aprendizaje con 20 puntos (mismo pipeline y CV que en la fase 13)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lc = learning_curve_data(
    pipeline, Xtr, ytr, cv=cv, n_points=20, scoring="roc_auc", n_jobs=1, random_state=42,
)
print("Puntos de la curva:", len(lc["train_sizes"]))
print("Primeros tamaños:", lc["train_sizes"][:5], "...")
print("Ultimos tamaños:", lc["train_sizes"][-3:])
print("Train ROC-AUC (media) ultimo punto:", round(lc["train_scores_mean"][-1], 4))
print("Val   ROC-AUC (media) ultimo punto:", round(lc["val_scores_mean"][-1], 4))

# Guardar en JSON para analisis posterior
import json as _json
lc_json = {
    "descripcion": "Curva de aprendizaje del modelo final RandomForest (pipeline completo) sobre train.",
    "metrica": "roc_auc",
    "n_total_train": int(len(Xtr)),
    "puntos": [
        {"n_train": int(s), "train_mean": round(float(tm), 4), "train_std": round(float(ts), 4),
         "val_mean": round(float(vm), 4), "val_std": round(float(vs), 4), "gap": round(float(tm - vm), 4)}
        for s, tm, ts, vm, vs in zip(lc["train_sizes"], lc["train_scores_mean"],
                                     lc["train_scores_std"], lc["val_scores_mean"], lc["val_scores_std"])
    ],
}
(ROOT / "reports" / "learning_curve.json").write_text(
    _json.dumps(lc_json, indent=2, ensure_ascii=False), encoding="utf-8")
print("Curva guardada en reports/learning_curve.json")


Puntos de la curva: 20
Primeros tamaños: [15, 31, 47, 63, 79] ...
Ultimos tamaños: [284, 300, 316]
Train ROC-AUC (media) ultimo punto: 0.9939
Val   ROC-AUC (media) ultimo punto: 0.8384
Curva guardada en reports/learning_curve.json



### 19.4.3 Representación gráfica y anatomía de la curva

Dibujamos la curva con **bandas de desviación** (media +/- 1σ entre folds) para train y
validation, marcando el criterio mínimo (0.75) y el baseline aleatorio (0.5). La figura
se guarda en `reports/figures/13_learning_curve.png`.

Con el dataset ampliado de 662 alumnos (train = 396 filas) la curva ya muestra el
patrón estándar suave: la validation sube de forma continua (~0.74 -> 0.845) y se
estabiliza en una meseta, sin el ruido brusco que producía el train de 228 filas.


In [9]:

# Figura de la curva de aprendizaje (20 puntos, media +/- desviacion)
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(lc["train_sizes"], lc["train_scores_mean"], "o-", color="#2c7bb6",
        label="Train (ROC-AUC)", linewidth=2, markersize=4)
ax.fill_between(lc["train_sizes"],
                np.array(lc["train_scores_mean"]) - np.array(lc["train_scores_std"]),
                np.array(lc["train_scores_mean"]) + np.array(lc["train_scores_std"]),
                alpha=0.15, color="#2c7bb6")
ax.plot(lc["train_sizes"], lc["val_scores_mean"], "s-", color="#d7191c",
        label="Validation (ROC-AUC)", linewidth=2, markersize=4)
ax.fill_between(lc["train_sizes"],
                np.array(lc["val_scores_mean"]) - np.array(lc["val_scores_std"]),
                np.array(lc["val_scores_mean"]) + np.array(lc["val_scores_std"]),
                alpha=0.15, color="#d7191c")
ax.axhline(0.75, ls="--", color="gray", label="Criterio minimo (0.75)")
ax.axhline(0.5, ls=":", color="black", label="Baseline aleatorio (0.5)")
ax.set_xlabel("Tamano del conjunto de entrenamiento (n de ejemplos)")
ax.set_ylabel("ROC-AUC (media +/- std, 5 folds)")
ax.set_title("Curva de aprendizaje del modelo final (20 puntos)")
ax.legend(loc="lower right")
ax.set_ylim(0.4, 1.0)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "13_learning_curve.png", dpi=120)
plt.show()


C:\Users\sgml1\AppData\Local\Temp\ipykernel_9428\1065596997.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



### 19.4.4 Verificación numérica del patrón estándar

Cuantificamos los indicadores de un buen entrenamiento:

1. **Convergencia de la curva de validation**: el ROC-AUC del último punto debe ser mayor
   que el del primero (la curva asciende) y el incremento entre el 90 % y el 100 % debe
   ser pequeño (meseta alcanzada).
2. **Gap train-validation estable**: en RandomForest el score de train es muy alto por el
   muestreo *bootstrap* (cada árbol se entrena con ~63 % de los datos con repetición), así
   que un gap de ~0.10-0.16 es **esperable y no indica overfitting**. Lo que delata
   overfitting real es que el gap **crezca** conforme aumentan los datos. Para medirlo de
   forma robusta comparamos el gap final con la **banda de ruido** (media +/- 2σ) de la zona
   estable: si cae dentro, no hay tendencia de crecimiento.
3. **Nivel absoluto**: el ROC-AUC de validation con el 100 % de los datos debe superar el
   criterio de aceptación (0.75), descartando underfitting.

También identificamos el **punto más alto** de la curva y comprobamos que la tendencia
posterior es plana (meseta), que es la lectura que pide el análisis: tras el máximo,
todo lo que sigue tiende hacia el valor de equilibrio.


In [10]:

# Verificacion numerica del patron estandar de la curva de aprendizaje
train_m = np.array(lc["train_scores_mean"])
val_m = np.array(lc["val_scores_mean"])
gaps = train_m - val_m
sizes = np.array(lc["train_sizes"])

# 1) La curva de validation asciende y alcanza meseta?
subida_total = val_m[-1] - val_m[0]
subida_final = val_m[-1] - val_m[-3]
print("1) Subida total de validation:", f"{subida_total:+.4f}", "(debe ser > 0)")
print("   Subida en la meseta final :", f"{subida_final:+.4f}", "(debe ser ~0)")

# 2) Gap train-validation: estable (no creciente) = sin overfitting real
#    (en RandomForest el gap ~0.1-0.16 es esperable por el bootstrap)
gap_final = gaps[-1]
gap_medio = float(np.mean(gaps[5:]))   # media de la zona estable (tras arranque)
gap_std = float(np.std(gaps[5:]))      # ruido de la zona estable
print("2) Gap train-validation final :", f"{gap_final:.4f}", "(RF espera ~0.10-0.16)")
print("   Gap medio (zona estable)   :", f"{gap_medio:.4f}", "+/-", f"{gap_std:.4f}")
print("   Banda de ruido (2 sigma)   : [", f"{gap_medio-2*gap_std:.4f}", ",", f"{gap_medio+2*gap_std:.4f}", "]")
print("   Gap final dentro de banda? :", "SI (sin tendencia de overfitting)" if gap_final <= gap_medio + 2*gap_std else "NO (posible overfitting)")

# 3) Nivel absoluto de validation (indicador de underfitting)
print("3) ROC-AUC validation con 100% datos:", f"{val_m[-1]:.4f}", "(criterio >= 0.75)")

# Punto mas alto y tendencia posterior
imax = int(np.argmax(val_m))
print()
print("Punto mas alto de validation: ROC-AUC =", f"{val_m[imax]:.4f}", "en n =", sizes[imax])
if imax < len(val_m) - 1:
    pendiente_post = float(np.polyfit(np.arange(imax + 1, len(val_m)), val_m[imax + 1:], 1)[0])
    print("Pendiente tras el maximo:", f"{pendiente_post:+.4f}",
          "(plana/negativa suave = meseta; positiva = seguiria subiendo)")

# Veredicto combinado
ok_subida = subida_total > 0.02
ok_meseta = abs(subida_final) < 0.02
ok_gap = gap_final <= gap_medio + 2*gap_std
ok_nivel = val_m[-1] >= 0.75
v_subida = "OK" if ok_subida else "REVISAR"
v_meseta = "OK" if ok_meseta else "REVISAR"
v_gap = "OK" if ok_gap else "REVISAR"
v_nivel = "OK" if ok_nivel else "REVISAR"
v_final = "PATRON ESTANDAR: modelo que generaliza" if all([ok_subida, ok_meseta, ok_gap, ok_nivel]) else "Revisar entrenamiento"
print()
print("Veredicto:")
print("  - Curva de validation asciende      : " + v_subida)
print("  - Meseta alcanzada (ultimos pasos)  : " + v_meseta)
print("  - Gap estable (sin overfitting)     : " + v_gap)
print("  - Sin underfitting (nivel >= 0.75)  : " + v_nivel)
print("  -> " + v_final)


1) Subida total de validation: +0.2076 (debe ser > 0)
   Subida en la meseta final : -0.0019 (debe ser ~0)
2) Gap train-validation final : 0.1555 (RF espera ~0.10-0.16)
   Gap medio (zona estable)   : 0.1653 +/- 0.0136
   Banda de ruido (2 sigma)   : [ 0.1382 , 0.1924 ]
   Gap final dentro de banda? : SI (sin tendencia de overfitting)
3) ROC-AUC validation con 100% datos: 0.8384 (criterio >= 0.75)

Punto mas alto de validation: ROC-AUC = 0.8450 en n = 268
Pendiente tras el maximo: -0.0010 (plana/negativa suave = meseta; positiva = seguiria subiendo)

Veredicto:
  - Curva de validation asciende      : OK
  - Meseta alcanzada (ultimos pasos)  : OK
  - Gap estable (sin overfitting)     : OK
  - Sin underfitting (nivel >= 0.75)  : OK
  -> PATRON ESTANDAR: modelo que generaliza



### 19.4.5 Curvas ROC y PR en test

La **curva ROC** representa TPR frente a FPR para todos los umbrales; la forma de un
buen modelo es la de una "bandera" (sube verticalmente y se aplana) con AUC muy por
encima de la diagonal (0.5 = azar). La **curva PR** es más informativa con desbalance
(39 % de positivos): su área debe superar con claridad la **línea de prevalencia**, que
es el rendimiento de un clasificador aleatorio en el espacio PR. Marcamos en ambas el
punto del umbral de coste congelado (estrella).


In [11]:

# Curva ROC en test con el punto del umbral de coste
roc = roc_curve_data(yte, p_cal)
fpr, tpr = roc["fpr"], roc["tpr"]
m = compute_metrics(yte, p_cal, threshold=umbral)

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot(fpr, tpr, color="#2c7bb6", linewidth=2.2,
        label="ROC-AUC = " + f"{m['roc_auc']:.3f}")
ax.plot([0, 1], [0, 1], "--", color="gray", label="Azar (AUC = 0.5)")
idx = np.argmin(np.abs(np.array(tpr) - m["recall"]))
ax.plot(fpr[idx], tpr[idx], "*", color="#d7191c", markersize=14,
        label="Umbral de coste (" + f"{umbral:.2f}" + ")")
ax.set_xlabel("FPR (1 - especificidad)")
ax.set_ylabel("TPR (sensibilidad)")
ax.set_title("Curva ROC en test")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "13_roc_test.png", dpi=120)
plt.show()


C:\Users\sgml1\AppData\Local\Temp\ipykernel_9428\575009635.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



#### Curva Precision-Recall

La **curva PR** complementa a la ROC cuando hay desbalance (39 % de positivos):
representa precision frente a recall para todos los umbrales. Un buen modelo mantiene la
precisión alta mientras sube el recall, y el área (PR-AUC) debe superar con claridad la
**línea de prevalencia** (proporción de positivos), que es el rendimiento de un
clasificador aleatorio en el espacio PR. Marcamos el punto del umbral de coste.


In [12]:

# Curva PR en test con la linea de prevalencia (rendimiento aleatorio)
pr = pr_curve_data(yte, p_cal)
precision, recall = pr["precision"], pr["recall"]
prevalencia = float(yte.mean())

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot(recall, precision, color="#d7191c", linewidth=2.2,
        label="PR-AUC = " + f"{m['pr_auc']:.3f}")
ax.axhline(prevalencia, ls="--", color="gray",
           label="Prevalencia (azar) = " + f"{prevalencia:.2f}")
idx_pr = np.argmin(np.abs(np.array(recall) - m["recall"]))
ax.plot(recall[idx_pr], precision[idx_pr], "*", color="#2c7bb6", markersize=14,
        label="Umbral de coste (" + f"{umbral:.2f}" + ")")
ax.set_xlabel("Recall (sensibilidad)")
ax.set_ylabel("Precision")
ax.set_title("Curva Precision-Recall en test")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "13_pr_test.png", dpi=120)
plt.show()


C:\Users\sgml1\AppData\Local\Temp\ipykernel_9428\22135237.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



### 19.4.6 Curva de calibración (cruda vs calibrada)

La **curva de calibración** (diagrama de fiabilidad) agrupa las predicciones en
intervalos y compara la probabilidad media predicha (X) con la frecuencia observada (Y).
Un modelo perfectamente calibrado cae sobre la diagonal; los puntos por encima indican
sobreconfianza. Comparamos las probabilidades **crudas** del RandomForest frente a las
**calibradas** con el calibrador sigmoidal (Platt) guardado en la fase 16: esperamos que
las calibradas se acerquen mucho más a la diagonal, con ECE más bajo.


In [13]:

# Curvas de calibracion: cruda vs calibrada (mismo test)
cal_raw = calibration_summary(yte, p_raw)
cal_cal = calibration_summary(yte, p_cal)

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot([0, 1], [0, 1], "--", color="gray", label="Calibracion perfecta")
ax.plot(cal_raw["prob_pred"], cal_raw["prob_true"], "o-", color="#999999",
        label="Cruda (Brier=" + f"{cal_raw['brier']:.3f}" + ", ECE=" + f"{cal_raw['ece']:.3f}" + ")")
ax.plot(cal_cal["prob_pred"], cal_cal["prob_true"], "s-", color="#2c7bb6",
        label="Calibrada (Brier=" + f"{cal_cal['brier']:.3f}" + ", ECE=" + f"{cal_cal['ece']:.3f}" + ")")
ax.set_xlabel("Probabilidad media predicha")
ax.set_ylabel("Frecuencia observada")
ax.set_title("Curva de calibracion en test (cruda vs calibrada)")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "13_calibracion.png", dpi=120)
plt.show()

print("Brier crudo   :", f"{cal_raw['brier']:.4f}", "| ECE crudo   :", f"{cal_raw['ece']:.4f}")
print("Brier calib   :", f"{cal_cal['brier']:.4f}", "| ECE calib   :", f"{cal_cal['ece']:.4f}")


Brier crudo   : 0.1931 | ECE crudo   : 0.1395
Brier calib   : 0.1868 | ECE calib   : 0.1292


C:\Users\sgml1\AppData\Local\Temp\ipykernel_9428\2151568510.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



### 19.4.7 Matriz de confusión y distribución de probabilidades

La **matriz de confusión** con el umbral congelado muestra dónde acierta y dónde falla
el modelo en test: TP (riesgo detectado), FP (alertas innecesarias), FN (riesgo no
detectado, el error más caro con coste 2) y TN. La **distribución de probabilidades**
por clase real (cruda y calibrada) muestra si el modelo separa bien los grupos y si la
calibración reparte las probabilidades por todo el rango [0,1] en lugar de amontonarlas
en los extremos. Marcamos la zona de abstención [0.30, 0.60).


In [14]:

# Matriz de confusion con el umbral de coste congelado
y_pred = (p_cal >= umbral).astype(int)
cm = confusion_matrix(yte, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["Bajo", "Alto"])
fig, ax = plt.subplots(figsize=(5.5, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Matriz de confusion (test, umbral=" + f"{umbral:.2f}" + ")")
ax.set_xlabel("TP=" + str(cm[1,1]) + " FP=" + str(cm[0,1]) + " FN=" + str(cm[1,0]) + " TN=" + str(cm[0,0]))
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "13_matriz_confusion.png", dpi=120)
plt.show()
print("Coste medio de decision:", f"{m['cost']:.4f}", "(FP=1, FN=2)")


Coste medio de decision: 0.3459 (FP=1, FN=2)


C:\Users\sgml1\AppData\Local\Temp\ipykernel_9428\3303218492.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



#### Distribución de probabilidades predichas

Un buen modelo debe **separar** las probabilidades de los alumnos con consumo alto
(rojo) de las de consumo bajo (azul): si ambas distribuciones se solaparan por completo,
el modelo no distinguiría. Además, la versión calibrada debe repartir las probabilidades
a lo largo de todo el rango [0,1] (en lugar de amontonarse en los extremos), lo que hace
que el umbral de coste sea significativo. Dibujamos los histogramas superpuestos para las
probabilidades crudas y calibradas, marcando la zona de abstención [0.30, 0.60).


In [15]:

# Distribucion de probabilidades por clase real (cruda y calibrada)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, p, titulo in zip(axes, [p_raw, p_cal], ["Crudas (RandomForest)", "Calibradas (sigmoidal)"]):
    ax.hist(p[yte == 0], bins=20, alpha=0.6, color="#4c72b0", label="Consumo bajo (real)")
    ax.hist(p[yte == 1], bins=20, alpha=0.6, color="#c44e52", label="Consumo alto (real)")
    ax.axvspan(0.30, 0.60, color="gray", alpha=0.15, label="Zona de abstencion")
    ax.axvline(umbral, color="black", ls="--", label="Umbral (" + f"{umbral:.2f}" + ")")
    ax.set_xlabel("Probabilidad predicha")
    ax.set_ylabel("Frecuencia")
    ax.set_title(titulo)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "13_distribucion_probabilidades.png", dpi=120)
plt.show()


C:\Users\sgml1\AppData\Local\Temp\ipykernel_9428\721709930.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



### 19.4.8 Resumen del diagnóstico de generalización

Recopilamos en una tabla los indicadores de evidencia y los comparamos con los
criterios del proyecto (fase 6.4). Todos deben cumplirse para afirmar que el
entrenamiento fue correcto y que el modelo **no** presenta overfitting ni underfitting.


In [16]:

# Tabla resumen de evidencia del buen entrenamiento
m_final = compute_metrics(yte, p_cal, threshold=umbral)

evidencia = pd.DataFrame({
    "Indicador": [
        "ROC-AUC (test)",
        "PR-AUC (test)",
        "F1 (umbral coste)",
        "Brier (calibrado)",
        "ECE (calibrado)",
        "Gap train-validation (curva 20 pts)",
        "Gap dentro de banda +/-2σ (sin overfitting)",
        "Subida validation (curva 20 pts)",
        "Coste medio de decision",
    ],
    "Valor": [
        f"{m_final['roc_auc']:.3f}",
        f"{m_final['pr_auc']:.3f}",
        f"{m_final['f1']:.3f}",
        f"{cal_cal['brier']:.4f}",
        f"{cal_cal['ece']:.4f}",
        f"{gap_final:.4f}",
        f"{gap_final:.4f} (banda +/-2σ: {gap_medio-2*gap_std:.3f}-{gap_medio+2*gap_std:.3f})",
        f"{subida_total:+.4f}",
        f"{m_final['cost']:.4f}",
    ],
    "Criterio": [
        ">= 0.75",
        "> prevalencia (0.39)",
        ">= 0.55",
        "<= 0.22",
        "<= 0.10",
        "~0.10-0.16 (RF)",
        "dentro de banda +/-2σ",
        "> 0 (asciende)",
        "minimo posible",
    ],
    "Cumple": [
        "SI" if m_final["roc_auc"] >= 0.75 else "NO",
        "SI" if m_final["pr_auc"] > prevalencia else "NO",
        "SI" if m_final["f1"] >= 0.55 else "NO",
        "SI" if cal_cal["brier"] <= 0.22 else "NO",
        "SI" if cal_cal["ece"] <= 0.10 else "NO",
        "SI" if ok_gap else "NO",
        "SI" if ok_gap else "NO",
        "SI" if subida_total > 0 else "NO",
        "-",
    ],
})
evidencia


,Indicador,Valor,Criterio,Cumple
0,ROC-AUC (test),0.766,>= 0.75,SI
1,PR-AUC (test),0.741,> prevalencia (0.39),SI
2,F1 (umbral coste),0.702,>= 0.55,SI
3,Brier (calibrado),0.1868,<= 0.22,SI
4,ECE (calibrado),0.1292,<= 0.10,NO
5,Gap train-validation (curva 20 pts),0.1555,~0.10-0.16 (RF),SI
6,Gap dentro de banda +/-2σ (sin overfitting),0.1555 (banda +/-2σ: 0.138-0.192),dentro de banda +/-2σ,SI
7,Subida validation (curva 20 pts),+0.2076,> 0 (asciende),SI
8,Coste medio de decision,0.3459,minimo posible,-



### 19.4.9 Conclusión del diagnóstico

Las evidencias gráficas confirman que el modelo **generaliza correctamente** dado el
tamaño de muestra disponible:

1. **Curva de aprendizaje (20 puntos)**: la validation asciende de forma continua
   (~0.74 -> 0.845) y alcanza una **meseta**; el punto máximo (0.845 en n=268) está
   dentro de la banda de ruido y la pendiente posterior es ~0 (tiende al equilibrio).
   El gap train-validation (~0.16) es el esperado en RandomForest por el bootstrap y se
   mantiene dentro de la banda +/-2σ -> **sin overfitting**; el nivel (0.838) supera el
   criterio 0.75 -> **sin underfitting**.
2. **Curva ROC**: forma de bandera con AUC ~ 0.77, muy por encima de la diagonal ->
   poder discriminativo real.
3. **Curva PR**: PR-AUC ~ 0.74, muy por encima de la línea de prevalencia (0.39) ->
   priorización útil de intervenciones con desbalance.
4. **Calibración**: la curva calibrada (sigmoidal) se acerca a la diagonal con ECE ~ 0.13.

**Diagnóstico**: el modelo final (RandomForest balanceado + calibración sigmoidal)
cumple los criterios de aceptación y su curva de aprendizaje sigue la trayectoria
estándar de un modelo bien entrenado, sin indicios de overfitting ni underfitting.
